## Quick smoke-test (CI-friendly)

Run the small smoke-test cell below (Cell 2) first to quickly validate that imports and DB access are working. This cell is intentionally fast and safe for CI: it does NOT start any servers or run long subprocesses. If this passes, proceed to the longer demo cells that start services.

In [6]:
# CI-friendly smoke test: quick import + DB sanity check
# - No subprocesses, no network calls
# - Fast check for CI or quick local verification
import sys, os, sqlite3
# ensure repo package importable from notebook
sys.path.append('..')
try:
    import app.gradio_main as gradio_main
    imported = True
except Exception as e:
    print('IMPORT_ERROR:', type(e).__name__, e)
    imported = False

db_path = os.path.join('..', 'data', 'agent.db')
if not os.path.exists(db_path):
    print('DB_MISSING:', db_path)
else:
    try:
        con = sqlite3.connect(db_path)
        cur = con.cursor()
        cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
        tables = {r[0] for r in cur.fetchall()}
        con.close()
        expected = {'audit_logs', 'judge_logs', 'students', 'human_labels'}
        missing = expected - tables
        if missing:
            print('DB_OK_BUT_MISSING_TABLES:', sorted(missing))
        else:
            print('SMOKE_TEST_PASSED: imports_ok=%s, db_tables_ok=True' % (imported,))
    except Exception as e:
        print('DB_ERROR:', type(e).__name__, e)


IMPORT_ERROR: ModuleNotFoundError No module named 'gradio'
SMOKE_TEST_PASSED: imports_ok=False, db_tables_ok=True


# Manual walkthrough: Student + Librarian POC

This notebook documents a complete manual walkthrough for the demo: start the POC and Admin UI, seed audit rows with the planner, approve an action, run the executor, and verify results. It also includes architecture and a short roadmap to full autonomy.


## Use case & objective

Use case: a librarian assistant that recommends books to students who haven't borrowed recently. The planner proposes recommendation actions and writes them to an audit log; a human librarian reviews and approves before execution.

Objective: provide a demo-safe, end-to-end flow showcasing an agentic planner with human oversight, an approval UI, and a safe executor that performs side-effects only after approval.


## High-level flow

1. Planner/Agent: observes catalog and student activity and writes candidate actions into `audit_logs` with status=`pending_approval`.
2. Admin (human): inspects candidate rows in the Admin UI and approves or rejects each action.
3. Executor: scans for `approved` rows, claims them atomically, performs the side-effect (simulated), and marks rows `executed` or `failed`.
4. Audit trail: notes are appended to each row to make the lifecycle auditable.

## Architecture (diagram)

The repository includes an SVG diagram at `docs/architecture.svg`. If your notebook renderer supports local SVGs it will display; otherwise open the file locally.

![architecture](./architecture.svg)

Notes: Audit DB is the single source of truth; the Admin UI enforces human-in-the-loop review to prevent unapproved side-effects.


## Is this an agentic app? (updated)

Short answer: It's a hybrid — intentionally cautious and auditable.

- Planner/Agent: the planner is agentic: it observes the catalog and student activity and proposes concrete actions (written to `audit_logs`).
- Judge (LLM evaluation): we've added an LLM-based judge scaffold (Hermes3 local server + HTTP fallback + deterministic heuristic). The judge scores candidate audit rows and persists results to `judge_logs` so you can trace automated assessments.
- Human-in-the-loop gate: by default a human librarian inspects `pending_approval` rows in the Admin UI and approves or rejects them. This prevents automatic side-effects.
- Optional cautious automation: set `AUTO_APPROVE_THRESHOLD` to a numeric value to auto-approve rows whose judge score meets the threshold (for example, very-high-confidence items). This allows a gradual move toward automation with clear guardrails.

Why this design? It gives the benefits of agentic observation and suggestion while preserving human control over actual side-effects. All decisions are recorded (`audit_logs`, `judge_logs`, optional `human_labels`) so you can evaluate the judge and gradually tune automation policies.

Key files: `tools/agent_job.py` (planner), `tools/llm_judge.py` (judge scaffold), `tools/admin_ui.py` (approval UI), `tools/executor.py` (idempotent execution).

## LLM & Embeddings: how a lightweight model helps (updated)

- Embedding-based retrieval (BGE): precompute embeddings for catalog items and student summaries; use vector similarity to find top candidates for recommendations (higher quality than TF-IDF). The system has optional guarded embedding + FAISS/Qdrant paths that are used only when the relevant libraries are installed.
- Re-ranking & copy generation: run a small LLM (Hermes3 or HTTP-based) to re-rank the top-K candidates and generate friendly blurb text for emails. The repo includes a Hermes3 run script and a guarded client that falls back to a deterministic heuristic when no LLM is available.
- LLM judge/eval: `tools/llm_judge.py` is a scaffold that calls Hermes3 (python client if installed, HTTP /v1/completions fallback), extracts a numeric `score` and `reason` from responses (robust parsing), and writes to `judge_logs` in `data/agent.db`. Admin UI surfaces judge scores and labels and optionally uses them to auto-approve highly confident rows.
- Safety checks: run a lightweight policy/classifier LLM to detect disallowed content or risky recommendations before auto-approval or execution. The judge is intended for triage, not as the single source of truth.
- Lightweight eval: record A/B labels in the DB and use `tools/eval_compare.py` to compare judge outputs vs human labels. This helps quantify judge precision/recall and tune thresholds.

Notes: Hermes3 integration is guarded — the system tries the python client first, then an HTTP POST to `HERMES3_URL` (default `http://127.0.0.1:11434/v1/completions`), and finally falls back to deterministic heuristics so the demo remains runnable without a local LLM.

## Step 1 — Start the Admin UI (and optional Student POC)

Start the Admin UI (default port 7861). The student-facing POC (the Student "book check out" UI) is an optional demo and — if you choose to run it — defaults to port 7883. The older reference to port 7684/7864 is obsolete and has been removed from this walkthrough.

Commands (background, logs to /tmp):

```bash
# Start Admin UI (required for librarian review)
.venv/bin/python tools/admin_ui.py 7861 &>/tmp/admin_ui.log & echo $!  # Admin UI PID printed

# Optional: start the Student POC (book checkout demo) on 7883
.venv/bin/python tools/run_gradio_poc.py 7883 &>/tmp/gradio_poc.log & echo $!  # Optional POC server PID printed
```

Verification (should show a listening Python process):

```bash
lsof -i :7861 -P -n || true
lsof -i :7883 -P -n || true    # optional POC listener
curl -sS http://127.0.0.1:7861/ | sed -n '1,40p'  # inspect the Admin root HTML (shows gradio config)
```

If startup fails, inspect logs:

```bash
sed -n '1,240p' /tmp/gradio_poc.log    # only if you started the optional POC
sed -n '1,240p' /tmp/admin_ui.log
```

Notes:
- The Admin UI is the canonical librarian-facing review interface for `audit_logs` rows and should be running for any human approval steps.
- The Student POC (book check out UI) is optional for demos and user testing; it is the most recently added use case and demonstrates a mock student login, weekly recommendations, and opt‑in persistence into `audit_logs` for requests and finished-book events.
- Use the optional POC when you want to demonstrate the student workflow end‑to‑end; keep it off for Admin-only testing or when simulating planner actions programmatically.

## Demo: Student 'Log a finished book' (persisted to audit_logs)

This short demo shows how the Student UI's 'Log a finished book' flow now persists a `return_book` action into `data/agent.db` as an `audit_logs` row with `status='pending_approval'`. The row is visible to the Admin UI and to the judge pipeline (if available).

Steps: 1) run the POC & Admin UI (see Step 1 above), 2) run the snippet below to simulate a student marking a book returned, 3) inspect the DB to see the inserted audit row.


In [ ]:
# Demo: mark a returned book and show the inserted audit_logs row
import sys, importlib, sqlite3, json
sys.path.append('agentic-rag-mvp')
m = importlib.import_module('app.gradio_main')
# Pick a demo book label from BOOK_DB (adjust if your BOOK_DB differs)
label = m._book_label(next(iter(m.BOOK_DB.keys())))
print('Using label:', label)
# Call the handler which now persists into data/agent.db
msg, choices = m._mark_request_returned('demo_student', 5, label)
print('Handler message:', msg)
# Inspect the most recent audit_logs row(s)
con = sqlite3.connect('agentic-rag-mvp/data/agent.db')
cur = con.cursor()
cur.execute("SELECT id, created_at, student_id, action_type, payload, status, notes FROM audit_logs ORDER BY id DESC LIMIT 5")
rows = cur.fetchall()
for r in rows:
    print('AUDIT ROW:', r)
con.close()


## Updated Architecture (Hermes3 judge + agentic planner)

The diagram below summarizes the updated architecture. Key additions: a local Hermes3 LLM server that can be used by the judge scaffold to score candidate audit rows, and the `judge_logs` table that records model outputs. The Admin UI surfaces judge labels/scores and supports an `AUTO_APPROVE_THRESHOLD` to optionally auto-approve very-high-confidence items.

(If your notebook renderer supports local SVGs, open `docs/architecture.svg` to view the updated diagram. Otherwise, here is a short textual summary:)

- Planner (tools/agent_job.py): observes catalog/student data → writes `audit_logs` (status=pending_approval).
- Judge (tools/llm_judge.py): reads `audit_logs` candidates → calls Hermes3 (python client or HTTP fallback) → writes `judge_logs` (audit_id, model, score, label, reason).
- Admin UI (tools/admin_ui.py): displays `audit_logs` + latest `judge_logs` info; humans approve/reject; supports auto-approve using `AUTO_APPROVE_THRESHOLD`.
- Executor (tools/executor.py): atomically claims `approved` rows and performs side-effects, then marks rows `executed` or `failed`.

You can launch Hermes3 (if available) with `scripts/run_hermes3.sh` and then run `tools/llm_judge.py --demo` to see judge outputs persist into `data/agent.db`.

### Inline Architecture Diagram

Below is the repository architecture diagram embedded inline. If your notebook renderer supports local SVGs it will display. If it does not render in the notebook viewer, open `docs/architecture.svg` in an image viewer.

![architecture](./architecture.svg)

This diagram highlights: Planner → audit_logs → Admin UI; the LLM Judge (Hermes3) that writes to `judge_logs`; and the Executor that claims `approved` rows. The notebook cells below exercise the judge and Admin UI programmatically.

In [ ]:
# Run the judge demo and compare judge_logs vs human_labels
import subprocess, sys, sqlite3, json, time, os

ROOT = 'agentic-rag-mvp'
DB = os.path.join(ROOT, 'data', 'agent.db')

# Run the judge demo script using the repo venv. This will add entries to judge_logs.
print('Running tools/llm_judge.py --demo (may take a few seconds)')
ret = subprocess.run([os.path.join('.venv','bin','python'), os.path.join(ROOT,'tools','llm_judge.py'), '--demo'], capture_output=True, text=True)
print('stdout:\n', ret.stdout[:2000])
print('stderr:\n', ret.stderr[:2000])

# Wait briefly for DB writes
time.sleep(1)

# Query judge_logs and human_labels to show a simple join for recent audit_ids
con = sqlite3.connect(DB)
cur = con.cursor()
cur.execute("""
SELECT j.audit_id, j.model, j.score, j.label, j.reason, h.label as human_label
FROM judge_logs j
LEFT JOIN human_labels h ON h.audit_id = j.audit_id
WHERE j.created_at >= datetime('now','-1 day')
ORDER BY j.created_at DESC
LIMIT 20
""")
rows = cur.fetchall()
print('\nRecent judge_logs (joined with human_labels if any):')
for r in rows:
    print(r)
con.close()


In [ ]:
# Programmatically start Admin UI, insert a persisted return, call Admin endpoints (refresh + approve), and verify DB change
import subprocess, time, os, sqlite3, requests, importlib, sys

ROOT = 'agentic-rag-mvp'
ADMIN_PORT = 7861
ADMIN_BASE = f'http://127.0.0.1:{ADMIN_PORT}'

# Start Admin UI in background using repo venv
print('Starting Admin UI (background)...')
admin_proc = subprocess.Popen([os.path.join('.venv','bin','python'), os.path.join(ROOT,'tools','admin_ui.py'), str(ADMIN_PORT)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
# give it a moment to start
time.sleep(2)

# Insert a persisted returned-book row using the app helper
sys.path.append(ROOT)
app_mod = importlib.import_module('app.gradio_main')
label = app_mod._book_label(next(iter(app_mod.BOOK_DB.keys())))
msg, choices = app_mod._mark_request_returned('auto_demo_student', 5, label)
print('Inserted via helper:', msg)

# Refresh pending via Admin API
print('Calling refresh_pending endpoint...')
try:
    resp = requests.post(f'{ADMIN_BASE}/gradio_api/run/refresh_pending', json={"data": []}, timeout=5)
    print('refresh_pending status:', resp.status_code)
    print(resp.text[:400])
except Exception as e:
    print('Could not call refresh_pending:', e)

# Approve the top-most pending id programmatically by extracting the id from DB
con = sqlite3.connect(os.path.join(ROOT,'data','agent.db'))
cur = con.cursor()
cur.execute("SELECT id FROM audit_logs WHERE status='pending_approval' ORDER BY id DESC LIMIT 1")
row = cur.fetchone()
if row:
    pending_id = row[0]
    print('Approving id:', pending_id)
    try:
        resp2 = requests.post(f'{ADMIN_BASE}/gradio_api/run/on_approve', json={"data": [str(pending_id), 'Approved from notebook demo']}, timeout=5)
        print('approve status:', resp2.status_code)
        print(resp2.text[:400])
    except Exception as e:
        print('Could not call on_approve:', e)
    # verify DB update
    cur.execute('SELECT id, status, notes FROM audit_logs WHERE id=?', (pending_id,))
    print('After approve:', cur.fetchone())
else:
    print('No pending rows found to approve')
con.close()

# Shutdown Admin UI process
print('Shutting down Admin UI process...')
admin_proc.terminate()
try:
    admin_proc.wait(timeout=5)
except Exception:
    admin_proc.kill()
print('Admin UI terminated.')


### Note about long-running / system cells

A few cells in this notebook start subprocesses (Admin UI) or call external services (Hermes3 judge) and may take several seconds to complete. Recommended safety tips before running them:

- Run these cells one at a time (not the whole notebook) so you can inspect outputs and logs between steps.
- Ensure the repository venv (`.venv`) is activated or present; many cells call `.venv/bin/python` to run scripts reliably.
- If a cell starts a server (Admin UI or Hermes3), wait for the service to become ready (you can `curl` the health endpoint or inspect the subprocess log). Increase `time.sleep()` in the cell if your machine is slow.
- If a cell fails to contact a local service, inspect `/tmp/admin_ui.log` or the service logs and re-run the cell after the service is healthy.
- Prefer the lightweight smoke-test cells (no subprocesses) when running in CI or restricted environments.

If you're unsure, run the small smoke-test cell first to validate DB access and module imports before running the longer cells.

In [ ]:
# Seed demo data (creates DB/tables if missing) and show pending audit rows
# This runs the agent_job seeder which inserts `pending_approval` rows.
import subprocess, os, sqlite3, time
ROOT = '..'
print('Running agent_job seeder (may create data/agent.db)')
ret = subprocess.run([os.path.join('.','venv','bin','python'), os.path.join(ROOT,'tools','agent_job.py')], capture_output=True, text=True)
print('stdout:\n', ret.stdout[:2000])
print('stderr:\n', ret.stderr[:2000])
# wait briefly for DB writes
time.sleep(0.5)
DB = os.path.join(ROOT, 'data', 'agent.db')
if os.path.exists(DB):
    con = sqlite3.connect(DB)
    cur = con.cursor()
    cur.execute("SELECT id, created_at, student_id, action_type, status FROM audit_logs WHERE status='pending_approval' ORDER BY id DESC LIMIT 20")
    rows = cur.fetchall()
    if rows:
        print('Pending audit rows:')
        for r in rows:
            print(r)
    else:
        print('No pending audit rows found')
    con.close()
else:
    print('DB not found at', DB)


## End-to-end: Manual approval walkthrough

This final section walks through an end-to-end test of the Admin UI + Judge workflow. It shows how to inspect a pending audit row (id 69 in the seeded demo), view the judge's raw JSON, approve the row, and verify the database was updated.

Follow these steps in order or run the cells below in a notebook environment where the project's virtualenv is active.

In [ ]:
# Inspect audit row id=69 and its judge logs
import sqlite3, json, os
DB = os.path.join('agentic-rag-mvp','data','agent.db')
con = sqlite3.connect(DB)
cur = con.cursor()
cur.execute('SELECT id, student_id, action_type, payload, status, notes, created_at FROM audit_logs WHERE id = ?', (69,))
row = cur.fetchone()
print('AUDIT ROW:\n', row)
print('\nJUDGE LOGS (latest first):')
cur.execute('SELECT id, model, score, label, reason, created_at FROM judge_logs WHERE audit_id = ? ORDER BY created_at DESC', (69,))
for r in cur.fetchall():
    print(r)
con.close()

In [ ]:
# Programmatically approve audit id=69 (same behavior as clicking Approve)
import sqlite3, os
DB = os.path.join('agentic-rag-mvp','data','agent.db')
con = sqlite3.connect(DB)
cur = con.cursor()
cur.execute('UPDATE audit_logs SET status = ?, notes = ? WHERE id = ?', ('approved', 'approved via notebook walkthough', 69))
con.commit()
# verify
cur.execute('SELECT id, status, notes FROM audit_logs WHERE id = ?', (69,))
print('After update:', cur.fetchone())
con.close()

In [ ]:
# Fetch and pretty-print the full judge JSON for audit id 69 (no UI needed)
from importlib import import_module
import json
m = import_module('agentic-rag-mvp.tools.admin_ui')
raw = m.fetch_full_judge_json(69)
if not raw:
    print('No judge JSON found for id 69')
else:
    try:
        obj = json.loads(raw)
        print(json.dumps(obj, indent=2, ensure_ascii=False))
    except Exception:
        # raw may already be a JSON string inside a string field
        print(raw)


### Note: programmatic approval == UI Approve

Running the programmatic approval cell above performs the same DB update as clicking the "Approve" button in the Admin UI. The UI's `on_approve` handler ultimately calls the same SQL update. Below is a tiny test that exercises the `on_approve` handler directly (invokes the function and verifies the DB row was updated). This is useful for CI or automated verification.

In [ ]:
# Small test: call the UI handler on_approve directly and verify DB update
from importlib import import_module
import sqlite3, os
m = import_module('agentic-rag-mvp.tools.admin_ui')
DB = os.path.join('agentic-rag-mvp','data','agent.db')
# reset row 69 to pending for the test
con = sqlite3.connect(DB)
cur = con.cursor()
cur.execute("UPDATE audit_logs SET status = ?, notes = ? WHERE id = ?", ('pending_approval', 'reset for test', 69))
con.commit()
con.close()
# call handler
res, _ = m.on_approve('69', 'approved via test')
print('handler returned:', res)
# verify
con = sqlite3.connect(DB)
cur = con.cursor()
cur.execute('SELECT id, status, notes FROM audit_logs WHERE id = ?', (69,))
print('after handler:', cur.fetchone())
con.close()

### Screenshot

If you want to include a screenshot of the Admin UI with id 69 selected, run the UI locally (http://127.0.0.1:7681), select id 69, take a screenshot, and drag the image into this notebook. Below is a placeholder where you can insert the image.

## Student Library POC — updated

This notebook previously described an earlier librarian preview POC. It has been updated: the demo now exposes a student‑facing Gradio POC (mock login + book request flows) and reuses the existing recommender logic. Key changes and how to run the demo:

- Where the code lives
  - Student POC: `tools/run_gradio_poc.py` (guarded: import-safe; only launches when run directly).
  - Planner/automated recommendations: `tools/agent_job.py` (writes `recommend_email` candidates to DB).
  - Librarian/Admin UI: `app/gradio_main.py` (review/approve pending audit rows).

- How the recommendation flow works now
  1. The planner (`tools/agent_job.py`) generates `recommend_email` candidates using the same recommender logic and inserts one or more rows into the shared SQLite DB (`agentic-rag-mvp/data/agent.db`) as `audit_logs` with `status='pending_approval'`.
  2. If a Judge is available, the planner runs `judge_candidates(...)` and may auto‑approve rows by updating `audit_logs.status = 'approved'` (controlled via judge logs + `AUTO_APPROVE_THRESHOLD`).
  3. The Student POC (`tools/run_gradio_poc.py`) when showing "This Week's Recommendations" prefers to display librarian‑approved `recommend_email` rows (SQL: SELECT ... WHERE action_type='recommend_email' AND status='approved'). If none exist for the student, it falls back to the internal recommender (TF‑IDF by default).

- TF‑IDF vs semantic embeddings
  - The POC defaults to TF‑IDF for speed and simplicity. There is an opt‑in checkbox in the UI labeled "Use semantic embeddings (opt-in)" to enable dense embedding retrieval (if the environment has sentence‑transformers/FAISS installed). This preserves the simple TF‑IDF path while allowing higher‑quality semantic lookups when explicitly requested.

- Student actions & persistence
  - Student actions ("Request Book", "Log Finished Book") are opt‑in for persistence. When the student checks the "Persist ... to audit_logs (opt-in)" box, the POC calls `persist_audit_row(student_id, action_type, payload)` which inserts a row into `audit_logs` with `status='pending_approval'`.
  - DB path: `agentic-rag-mvp/data/agent.db`.
  - `audit_logs` key columns: `id, created_at, student_id, action_type, payload (JSON), status, notes`.

- How to run the student POC locally
  1. From the repo root, ensure venv activated and minimal deps installed:
     ```bash
     .venv/bin/python -m pip install pandas numpy gradio scikit-learn
     # optional for semantic embeddings:
     .venv/bin/python -m pip install sentence-transformers faiss-cpu
     ```
  2. Start the POC (defaults to port 7883):
     ```bash
     .venv/bin/python agentic-rag-mvp/tools/run_gradio_poc.py 7883
     ```
  3. Open http://127.0.0.1:7883 and try:
     - Enter a Student ID (e.g., `S1`) and click "Refresh Weekly Recs" — approved `recommend_email` rows (if present) will be shown; otherwise TF‑IDF recommendations are shown.
     - Request a book and check the persist opt‑in checkbox to create a `request_book` audit row.
     - Log a finished book with opt‑in to create a `finished_book` audit row.

- Notes / cleanup done
  - Obsolete references to the old preview port and non‑student UI were removed from this section. The notebook now points to the student POC, the planner, and the Librarian UI as the canonical places to look.

If you want, I can also:
- Update other cells in this notebook to include a tiny code cell that programmatically queries `agentic-rag-mvp/data/agent.db` and prints the latest `audit_logs` rows, or
- Add a short end‑to‑end demo cell that runs `tools/agent_job.py` and then shows how the student POC reads approved recommendations.

In [ ]:
# Demo: TF-IDF fallback verification (runs in the notebook kernel)
# This cell calls the POC recommender with use_semantic=False to show the TF-IDF path
import sys, importlib
# Ensure repo is importable from the docs folder
sys.path.insert(0, '..')

try:
    from tools import run_gradio_poc as poc
except Exception as e:
    print('COULD_NOT_IMPORT_POC:', type(e).__name__, e)
    raise
,
,
,
,
,
,
,
,
,
,
,
,
,

: 
,
: 
,
: {
: 

: [
,
,
,
,
,
  - When to use: local demos, manual student-facing testing, and small-scale UX checks.

- `tools/agent_job.py` (Planner / Scheduler)
  - Purpose: periodic planner that observes catalog and student activity, generates candidate actions (e.g., `recommend_email`) using the recommender, and writes them to `data/agent.db` `audit_logs` as `pending_approval` rows.
  - Behavior: after inserting rows it calls the Judge (if available) via `judge_candidates(...)`, and can auto‑approve rows based on judge scores + `AUTO_APPROVE_THRESHOLD`.
  - When to use: run periodically (cron/APS) to surface recommendations; also useful as a one-off seeder for demos.

- `tools/llm_judge.py` (Judge scaffold)
  - Purpose: score and label candidate audit rows using an LLM (Hermes3 local client or HTTP fallback). Writes results into `judge_logs` (audit_id, model, score, label, reason).
  - Intended application: triage and quality scoring of planner candidates; supports safe auto‑approve when configured.
  - Notes: guarded with fallbacks and heuristics to remain demo‑safe when a local LLM is unavailable.

- `app/gradio_main.py` (Admin / Librarian UI)
  - Purpose: librarian-facing Gradio UI to review pending `audit_logs` rows, approve/reject, add notes, and manage campaigns/seed books. Exposes programmatic endpoints the notebook can call (refresh, approve).
  - When to use: human-in-the-loop review for production or demos.

- `tools/admin_ui.py` (legacy admin runner) / start scripts
  - Purpose: launcher for the Admin UI (some setups use a separate runner script). Use the Admin UI to approve items created by the planner or students.

- `tools/executor.py` (Executor / Worker)
  - Purpose: claims `approved` rows atomically and performs side-effects (email sends, updates, external APIs). Marks rows `executed` or `failed` and appends execution notes. Designed to be idempotent and safe for concurrent workers.
  - When to use: run after approval to perform intended side-effects; supports dry-run for safety checks.

- `app/vector_store.py`, `tools/eval_compare.py`, and optional scripts
  - Purpose: vector store helpers (Qdrant/FAISS) and evaluation utilities (compare judge outputs vs human labels). Use them to integrate embeddings and evaluate judge performance.

- `data/agent.db` (SQLite) — single source of truth
  - Tables of interest: `audit_logs`, `students`, `judge_logs`, `human_labels` (optional). All candidate actions, student requests, judge scores, and human labels persist here.

### How these tools orchestrate together (compact)
1. Planner (`tools/agent_job.py`) -> writes `audit_logs` (status=`pending_approval`).
2. Judge (`tools/llm_judge.py`) -> scores candidates and writes `judge_logs`.
3. Admin UI (`app/gradio_main.py`) -> humans inspect `pending_approval` rows and approve/reject (updates `audit_logs.status`).
4. Executor (`tools/executor.py`) -> claims `approved` rows, performs side-effects, marks `executed` or `failed`.
5. Student POC (`tools/run_gradio_poc.py`) -> displays `recommend_email` rows with `status='approved'`; students may also create `request_book` or `finished_book` `audit_logs` rows (opt‑in).

### Is this an "agentic" app? (clarified)
- Short answer: partially — the planner component is agentic (it *proposes* actions autonomously), but actual side‑effects are gated by human approval (Admin UI) or by conservative auto‑approval policies.
- What is agentic:
  - The planner observes data and autonomously generates concrete candidate actions and writes them to the audit trail. That is the agentic part — it acts as an automated observer and proposer.
- What is not agentic (by default):
  - Execution of side‑effects (emails, external API calls) is not automatic — it requires human approval in the Admin UI. This keeps operations auditable and prevents runaway automation.
- Optional controlled automation:
  - You can enable partial automation by setting `AUTO_APPROVE_THRESHOLD` and running a Judge that reliably scores items. Rows whose judge score ≥ threshold can be auto‑moved to `approved`, enabling the Executor to run them without an explicit human UI interaction. Use this cautiously and monitor `judge_logs` + `human_labels`.

### Specific application of the LLM in this system
- Judge / triage:
  - Primary use: evaluate candidate audit rows (quality, safety, suitability) and output a numeric `score` plus short `reason`/label saved to `judge_logs`. The planner uses these scores to optionally auto‑approve high‑confidence items.
- Re‑ranking & blurb generation:
  - Optional secondary use: a small LLM or local re‑ranker can re‑score top‑K candidates and generate friendly copy (email subject, one‑line blurb) for librarian review.
- Safety & PII filters:
  - The LLM is also used as a heuristic filter for disallowed content or risky recommendations before they reach librarians (detects PII, questionable content). If Hermes3 is unavailable, the system falls back to deterministic heuristics.
- Operational notes:
  - Hermes3 is the preferred local LLM in the repo (guarded client + HTTP fallback). All LLM calls are wrapped with fallbacks and robust parsing so the demo remains runnable without a local LLM.

If you'd like, I can now:
- Insert a small verification cell that runs `tools/agent_job.py --dry-run` and prints the planned actions without modifying the DB, or
- Add a concise YAML or README snippet documenting how to deploy these tools in a small pilot (commands, cron entry, monitoring pointers).

In [ ]:
# End-to-end demo: planner -> (judge) -> approve -> student view
# This cell runs the planner (agent_job), persists recommend_email candidates to the DB,
# then programmatically approves them and shows the Student POC view for a demo student.

import sys, os, importlib, sqlite3, json
sys.path.insert(0, 'agentic-rag-mvp')

# Import the agent_job module (planner) and the POC recommender/catalog
from tools import agent_job as aj
from tools import run_gradio_poc as poc

# Initialize DB and seed students (idempotent)
con = aj.init_db()
aj.seed_students(con)

# Plan actions (uses poc.recommend_by_student under the hood)
actions = aj.plan_actions(con)
print('Planned actions for students:', len(actions))

# Persist planner actions into audit_logs (this will call the judge if available)
inserted_ids = aj.persist_actions(con, actions)
print('Inserted audit row ids:', inserted_ids)

# Choose a demo student to inspect (use one from the POC borrow_history)
demo_student = 'S1'
cur = con.cursor()

# Show pending recommend_email rows for the demo student
cur.execute("SELECT id, payload, status FROM audit_logs WHERE student_id = ? AND action_type = 'recommend_email' ORDER BY id DESC", (demo_student,))
pending = cur.fetchall()
print(f'Found {len(pending)} recommend_email rows for', demo_student)
for r in pending:
    print(r)

# Approve all pending recommend_email rows for the demo student to simulate librarian approval
pending_ids = [r[0] for r in pending if r[2] == 'pending_approval']
if pending_ids:
    cur.executemany('UPDATE audit_logs SET status = ?, notes = ? WHERE id = ?', [( 'approved', 'demo-approved', pid) for pid in pending_ids])
    con.commit()
    print('Approved ids:', pending_ids)
else:
    print('No pending rows to approve for', demo_student)

# Build the student-facing recommendations view: prefer approved audit_logs rows
cur.execute("SELECT id, payload FROM audit_logs WHERE student_id = ? AND action_type = 'recommend_email' AND status = 'approved' ORDER BY id DESC LIMIT 10", (demo_student,))
approved = cur.fetchall()
student_view = []
if approved:
    for aid, payload in approved:
        try:
            j = json.loads(payload)
            bid = int(j.get('book_id'))
            if bid in poc.book_index:
                row = poc.catalog.iloc[poc.book_index[bid]]
                student_view.append({'audit_id': aid, 'book_id': bid, 'title': row['title'], 'score': j.get('score')})
        except Exception:
            continue
else:
    print('No approved recommendations found; falling back to recommender (TF-IDF)')
    recs = poc.recommend_by_student(student_id=demo_student, top_k=5, use_semantic=False)
    for bid, score, row in recs:
        student_view.append({'book_id': bid, 'title': row['title'], 'score': score})

print('\nStudent view for', demo_student)
for it in student_view:
    print(it)

# Optionally summarize the DB (from agent_job)
aj.summarize(con)
con.close()

## Architecture (high level)

```mermaid
flowchart LR
  subgraph DataSources
    Catalog[(Card Catalog CSV/JSON)]
    Borrow[(Borrow History Logs)]
  end
  Catalog --> Ingest[Ingestion + Normalization]
  Borrow --> Ingest
  Ingest --> DB[(Central SQLite / Postgres for scale)]
  Ingest --> Analytics[Basic Analytics / Dashboards]
  DB --> Recommender[Recommender Service]
  Recommender --> Judge[LLM Judge (hermes3, + HTTP/heuristic fallback)]
  Recommender --> AdminUI[Admin UI (librarian review - Gradio/React)]
  Judge --> DB
  AdminUI --> DB
  AdminUI -->|approve/reject| DB
  StudentUI[Student Discovery POC] -->|optional| Recommender
  style Judge fill:#f9f,stroke:#333,stroke-width:1px
  style Recommender fill:#def,stroke:#333,stroke-width:1px
  style AdminUI fill:#ffd,stroke:#333,stroke-width:1px
```

Notes:
- The `Recommender` produces candidate recommendations per student or cohort.
- Each candidate recommendation is written to an `audit_logs` table as a pending action.
- The `Judge` (hermes3 by default) scores and labels recommendations to help prioritize high-confidence suggestions and enable an auto-approve path.
- The `Admin UI` is the librarian-facing review interface where librarians can approve, reject, add notes, or edit recommendations before they reach students.
- `StudentUI` is optional for pilot phases; the initial delivery focuses on librarian workflows.

## Phased Rollout Plan

1) Discovery & Data Ingestion (2-4 weeks)
- Work with district librarians to get sample card catalog (CSV/JSON) and borrow logs.
- Define minimal ingestion schema and mapping rules (normalize titles, authors, IDs).
- Run an initial ingestion to create a catalog and low-fidelity student profiles.

2) Pilot Recommender + Librarian Review (4-6 weeks)
- Deploy the `Recommender` service in a pilot environment.
- Connect the `Admin UI` (librarian-facing) for review of candidate recommendations.
- Run a 2-week pilot in 1-2 schools; collect librarian feedback and acceptance rates.

3) Judge & Auto-Approve Tuning (2-3 weeks)
- Integrate `hermes3` as the LLM judge for scoring suggestions.
- Choose an AUTO_APPROVE_THRESHOLD and run A/B tests to calibrate.
- Add logging and analytics for wrong approvals and librarian overrides.

4) Student Discovery Pilot (2-4 weeks)
- Add the `StudentUI` (POC) in a test classroom; monitor engagement and borrow rates.

5) Rollout & Ongoing Operations (ongoing + 1-2 months ramp)
- Train librarians, deploy to more schools, and iterate on UX and rules.
- Implement data governance, backups, and incident playbooks.

Total estimated initial engagement: 10–16 weeks for a full pilot to student-facing rollout (excluding long-term scaling).

## Team & Roles (example staffing)

- Engagement Lead (consulting partner) — overall sponsor and client liaison.
- Technical Lead (you) — architecture, delivery, code reviews.
- Data Engineer (1) — ingestion pipelines, schema mapping, DB ops.
- ML/LLM Engineer (1) — recommender models, hermes3 integration, fallback heuristics.
- Frontend Developer (1) — Admin UI improvements and StudentUI work.
- UX / Librarian Liaison (1) — conduct librarian interviews, usability testing.
- QA / Tester (0.5) — acceptance tests for data flows and UI.

For a small pilot, a compact team of 4 (Technical Lead, Data Engineer, ML Engineer, UX Liaison) plus fractional QA is sufficient. For district-wide rollout add a devops engineer and more front-end capacity.

## Data & Privacy Considerations

- Keep student-identifying data minimal in the recommendation workflow. Use anonymized student IDs where possible and store PII separately with strict access controls.
- Ensure the ingestion process is run in a secure environment; prefer district-run compute or a secure cloud tenant.
- Maintain an audit trail (`audit_logs`, `judge_logs`, `human_labels`) for every decision and human action to support explainability and accountability.
- Provide librarians with explicit controls to redact or remove entries tied to specific students on request.

Regulatory note: ensure compliance with local regulations (COPPA, FERPA, or country-specific student privacy laws).

## Success Metrics & Monitoring

- Engagement: % of students issued recommendations who borrow at least one recommended book (primary metric).
- Librarian acceptance rate: fraction of recommendations approved by librarians.
- Lift in books read: average books read per student pre/post pilot.
- False positives / bad recommendations: tracked via librarian overrides and human labels.
- System health: ingestion latency, judge latencies, error rates, and DB size.

Monitoring: integrate simple dashboards (Grafana / Metabase) for operational metrics and weekly review reports for librarians and partners.

## Risks & Mitigations

- Risk: LLM judge returns empty or misleading responses (we observed empty `choices[0].text` in historical `judge_logs`).
  - Mitigation: fallback heuristics and HTTP fallback; flag empty responses in UI and store raw_response for post-hoc analysis.

- Risk: Librarians distrust automated suggestions.
  - Mitigation: Keep librarians in the loop; make UI review primary; highlight why a suggestion was made (simple explanation features).

- Risk: Data quality issues in catalog/borrow logs.
  - Mitigation: provide data validation tooling during ingestion and a simple dashboard for anomalies.

- Risk: Scope creep / long procurement cycles.
  - Mitigation: keep initial deliverable small (catalog + librarian review + judge), show impact with a 2-week pilot, and provide clear next steps for scale.